# 03 - Baselines

Computes the three Model 0 baselines (monthly climatology, persistence,
linear interpolation) against the canonical artificial-gap pool and scores
them. Fully executable on the public data included in this repository.


In [ ]:
import pandas as pd
import sys
sys.path.insert(0, "../src")
from coastal_gap_reconstruction.data_loading import load_daily_target, load_validation_gap_pool
from coastal_gap_reconstruction.artificial_gap_validation import apply_artificial_gap
from coastal_gap_reconstruction.baseline_imputation import run_all_baselines
from coastal_gap_reconstruction.scoring_metrics import compute_gap_metrics, aggregate_metrics

target_df = load_daily_target("../data_public/chlorophyll/chlorophyll_daily_target.csv")
gap_pool = load_validation_gap_pool("../data_public/chlorophyll/chlorophyll_validation_gaps.csv")
print(len(gap_pool), "gaps loaded")


## Run baselines on every gap in the pool

In [ ]:
all_metrics = []

for _, g in gap_pool.iterrows():
    start = pd.Timestamp(g["start_date"])
    gap_length = int(g["gap_length"])

    masked = apply_artificial_gap(target_df, start, gap_length)
    predictions = run_all_baselines(masked, start, gap_length)

    metrics = compute_gap_metrics(
        target_df=target_df,
        predictions=predictions,
        start_date=start,
        gap_length=gap_length,
        gap_id=g["gap_id"],
        gap_info=g.to_dict(),
    )
    all_metrics.extend(metrics)

metrics_df = pd.DataFrame(all_metrics)
metrics_df.head()


## Aggregate by method and gap length

In [ ]:
summary = aggregate_metrics(metrics_df, groupby_cols=["method", "gap_length"])
summary.sort_values(["method", "gap_length"])


## Aggregate by method only

In [ ]:
overall = aggregate_metrics(metrics_df, groupby_cols=["method"])
overall


## Interpretation

These three baselines establish the floor for this benchmark. Linear
interpolation typically performs best among the three for short gaps (it
has access to both edges of the gap), but is not forecast-safe. Compare
these numbers against `results_public/chlorophyll/chlorophyll_benchmark_summary.csv`
for how more complex methods (engineered tabular/gap-edge models,
TS-ICL) perform relative to these baselines.
